## Add hexapod motions to the mount plots

Craig Lage  04-Jun-25

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
import lsst.summit.utils.butlerUtils as butlerUtils
from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData
from lsst.summit.utils.simonyi.mountData import getAzElRotHexDataForPeriod, getAzElRotHexDataForExposure
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId

In [ ]:
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
expId = 2026031000072
dataId1 = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord1 = getExpRecordFromDataId(butler, dataId1)
expId = 2026031000073
dataId2 = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord2 = getExpRecordFromDataId(butler, dataId2)
expId = 2026031000074
dataId3 = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord3 = getExpRecordFromDataId(butler, dataId3)

In [ ]:
data = getAzElRotHexDataForPeriod(client, expRecord1.timespan.begin, expRecord3.timespan.end)

In [ ]:
fig, axes = plt.subplots(3,2,figsize=(15,10), sharex=True)
plt.subplots_adjust(hspace=0)
plt.suptitle("TMA motions 20260310, images 72, 73, 74\nRed and Green dotted lines are shutter open/close")
ax = axes[0][0]
data.azimuthData.actualPosition.plot(ax = ax)
ax.set_ylabel("Azimuth position\n(degrees)")
ax = axes[1][0]
data.azimuthData.actualVelocity.plot(ax = ax)
ax.set_ylabel("Azimuth velocity\n(degrees/sec)")
ax = axes[2][0]
data.azimuthData.actualAcceleration.plot(ax = ax)
ax.set_ylabel("Azimuth acceleration\n(degrees/sec^2)")
ax = axes[0][1]
data.elevationData.actualPosition.plot(ax = ax)
ax.set_ylabel("Elevation position\n(degrees)")
ax = axes[1][1]
data.elevationData.actualVelocity.plot(ax = ax)
ax.set_ylabel("Elevation velocity\n(degrees/sec)")
ax = axes[2][1]
data.elevationData.actualAcceleration.plot(ax = ax)
ax.set_ylabel("Elevation acceleration\n(degrees/sec^2)")
for ax in axes.flatten():
    for expRecord in [expRecord1, expRecord2, expRecord3]:
        ax.axvline(expRecord.timespan.begin.utc.isot, ls='--', color='green')
        ax.axvline(expRecord.timespan.end.utc.isot, ls='--', color='red')
fig.savefig(f"/home/c/cslage/u/MTMount/mount_plots/Mount_motions_20260310, 72-74.png", 
            bbox_inches='tight', pad_inches=1.2)

In [ ]:
axes.flatten()

In [ ]:
expId = 2026031200203
dataId1 = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord1 = getExpRecordFromDataId(butler, dataId1)
data1 = getAzElRotHexDataForExposure(client, expRecord1)
expId = 2026031200241
dataId2 = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord2 = getExpRecordFromDataId(butler, dataId2)
data2 = getAzElRotHexDataForExposure(client, expRecord2)

In [ ]:
data1.

In [ ]:
hexNames = ["X", "Y", "Z", "U", "V", "W"]
datNames = ['M2', 'Cam']
values = np.zeros([2, 2, 5])
for k, data in enumerate([data1, data2]):
    for j, dat in enumerate([data.m2hexData, data.camhexData]):
        for i in [2, 0, 1, 3, 4]:
            value = np.median(dat[f'position{i}'].values)
            if i in [3,4]:
                value *=3600
            values[k, j, i] = value
            counter += 1
print(f"DOF name\t\tDOF-203\t\tDOF-241\t\tDiff\t\t%Diff")
for j in range(2):
    for i in [2, 3, 4]:
        line = f"{datNames[j]}-{hexNames[i]}\t\t"
        for k in range(2):
            line += f"{values[k,j,i]:10.3f}\t\t"
        diff = values[1,j,i] - values[0,j,i]
        pct = (values[1,j,i] - values[0,j,i]) / values[0,j,i] * 100.0
        line += f"{diff:10.3f}\t\t {pct:10.3f}"
        print(line)
            
    

In [ ]:
values

In [ ]:
i = 3
np.median(dat[f'position{i}'].values)